# Study 922 — Floating-Rate Front End — the teardown

The pairwise race with HAC *t*, the launch-liquidity era cut, the ^IRX regime split, the HAC-OLS rate-direction contrast and its classifier sweep, the cycle windows, the duration attribution, block-bootstrap CIs, and the borrow / cost / cash-proxy sweeps. Every real number is frozen from `docs/results.md` (fingerprint `4bfc2a85745c`); the live cells are the synthetic control.

**Conventions.** Daily total-return closes (`auto_adjust=True`); ^IRX is a price-only yield index and is never held. Exactly one execution lag: every ^IRX-derived object (regime label, cash accrual) is formed through day *t* and applied at *t*+1. Sleeves are held, not traded, so cost is a round trip amortised over the holding horizon; read as a dollar-neutral pair, the short leg pays borrow. Survivorship: no cross-section — four named, still-listed funds.

In [1]:
R = {'start': '2014-02-04', 'end': '2026-06-30', 'n_days': 3118, 'fp': '4bfc2a85745c', 'usfr_ret': 1.91, 'usfr_vol': 1.46, 'usfr_dd': -2.12, 'usfr_exc': 0.029, 'usfr_sh': 0.02, 'tflo_ret': 1.95, 'tflo_vol': 1.98, 'tflo_dd': -5.01, 'tflo_exc': 0.067, 'tflo_sh': 0.03, 'bil_ret': 1.75, 'bil_vol': 0.26, 'bil_dd': -0.24, 'bil_exc': -0.131, 'bil_sh': -0.56, 'shy_ret': 1.48, 'shy_vol': 1.47, 'shy_dd': -5.71, 'shy_exc': -0.406, 'shy_sh': -0.28, 'usfr_bil': 0.16, 't_usfr_bil': 0.68, 'tflo_bil': 0.198, 't_tflo_bil': 0.96, 'usfr_shy': 0.435, 't_usfr_shy': 0.96, 'bil_shy': 0.275, 't_bil_shy': 0.72, 'usfr_tflo': -0.038, 't_usfr_tflo': -0.13, 'early_usfr_bil': 0.169, 'early_t': 0.23, 'late_usfr_bil': 0.155, 'late_t': 1.4, 'late_tflo_bil': 0.143, 'late_tflo_t': 2.08, 'late_usfr_shy': 0.693, 'late_usfr_shy_t': 1.24, 'reg_rise_n': 626, 'reg_flat_n': 2038, 'reg_fall_n': 390, 'rise_usfr': 2.65, 'rise_bil': 2.12, 'rise_shy': -0.04, 'rise_gap': 2.69, 'rise_gap_t': 1.94, 'flat_usfr': 1.55, 'flat_bil': 1.49, 'flat_shy': 1.73, 'flat_gap': -0.18, 'flat_gap_t': -0.35, 'fall_usfr': 3.14, 'fall_bil': 2.84, 'fall_shy': 2.87, 'fall_gap': 0.26, 'fall_gap_t': 0.23, 'contrast': 2.43, 'contrast_t': 1.37, 'rising_extra': 2.87, 'rising_extra_t': 1.92, 'falling_extra': 0.45, 'falling_extra_t': 0.37, 'contrast_bil': 2.19, 'contrast_bil_t': 1.33, 'sweep_positive': 12, 'sweep_total': 12, 'sweep_fire': 0, 'sweep_lo': 0.03, 'sweep_hi': 6.41, 'sweep_best': 2.69, 'sweep_best_t': 1.94, 'zirp_usfr': 0.73, 'zirp_bil': 0.58, 'zirp_shy': 1.11, 'zirp_gap': -0.38, 'zirp_gap_t': -0.8, 'hike_usfr': 3.39, 'hike_bil': 2.85, 'hike_shy': -1.07, 'hike_gap': 4.46, 'hike_gap_t': 1.99, 'hike_usfr_bil': 0.54, 'hike_usfr_bil_t': 2.05, 'hike_n': 356, 'plat_usfr': 4.83, 'plat_bil': 5.24, 'plat_shy': 5.5, 'plat_gap': -0.66, 'plat_gap_t': -0.4, 'cut_usfr': 4.24, 'cut_bil': 4.07, 'cut_shy': 3.39, 'cut_gap': 0.86, 'cut_gap_t': 0.87, 'usfr_dd_liq': -0.4, 'tflo_dd_liq': -0.16, 'bil_dd_liq': -0.21, 'shy_dd_liq': -5.71, 'usfr_vol_liq': 0.6, 'shy_vol_liq': 1.67, 'ci_usfr_bil_lo': -0.244, 'ci_usfr_bil_hi': 0.548, 'ci_usfr_bil_neg': 21.4, 'ci_usfr_shy_lo': -0.432, 'ci_usfr_shy_hi': 1.321, 'ci_usfr_shy_neg': 15.0, 'cost1_1y': 0.395, 'cost5_1y': 0.235, 'cost5_3y': 0.368, 'borrow25': 0.185, 'borrow50': -0.065, 'borrow100': -0.565, 'bil_sh_360': 1.836, 'hike_dirx': 4.94, 'hike_gap_cum': 6.45, 'hike_pred': 9.13, 'cut_dirx': -1.24, 'cut_gap_cum': 1.68, 'cut_pred': -2.29, 'syn_contrast': 8.35, 'syn_t': 14.2, 'syn_rise': 5.43, 'syn_fall': -2.92, 'syn_null_mean': 0.14, 'syn_null_sd': 0.4, 'syn_null_fire': 0}

## 1. The four sleeves, and the excess-of-cash caveat

Excess-of-cash uses the ^IRX/252 accrual **proxy**. Note how badly the *levels* behave: BIL's excess Sharpe is −0.56 on a 252 basis and **+1.84** on 360, because a few bps of convention dominate a 0.26%-vol series. Pairwise differences are invariant to the convention (cash cancels), which is why they are the headline.

In [2]:
print(f"{'fund':5s} {'ret':>8s} {'vol':>7s} {'maxDD':>8s} {'excess':>9s} {'exSharpe':>9s}")
for k in ('usfr', 'tflo', 'bil', 'shy'):
    print(f"{k.upper():5s} {R[k+'_ret']:+7.2f}% {R[k+'_vol']:6.2f}% {R[k+'_dd']:+7.2f}% "
          f"{R[k+'_exc']:+8.3f}% {R[k+'_sh']:+9.2f}")
print(f"\ncash-proxy fragility: BIL excess Sharpe {R['bil_sh']:+.2f} on IRX/252 "
      f"vs {R['bil_sh_360']:+.2f} on IRX/360 -> report differences, not levels")

fund       ret     vol    maxDD    excess  exSharpe
USFR    +1.91%   1.46%   -2.12%   +0.029%     +0.02
TFLO    +1.95%   1.98%   -5.01%   +0.067%     +0.03
BIL     +1.75%   0.26%   -0.24%   -0.131%     -0.56
SHY     +1.48%   1.47%   -5.71%   -0.406%     -0.28

cash-proxy fragility: BIL excess Sharpe -0.56 on IRX/252 vs +1.84 on IRX/360 -> report differences, not levels


## 2. The pairwise race — no unconditional winner clears |*t*| = 2

In [3]:
pairs = [('USFR-BIL', R['usfr_bil'], R['t_usfr_bil']),
         ('TFLO-BIL', R['tflo_bil'], R['t_tflo_bil']),
         ('USFR-SHY', R['usfr_shy'], R['t_usfr_shy']),
         ('BIL-SHY',  R['bil_shy'],  R['t_bil_shy']),
         ('USFR-TFLO', R['usfr_tflo'], R['t_usfr_tflo'])]
for p, d, t in pairs:
    flag = '  <- |t|>=2' if abs(t) >= 2 else ''
    print(f'{p:10s} {d:+.3f}%/yr   HAC t = {t:+.2f}{flag}')
print(f"\nbootstrap (21d blocks): USFR-BIL 95% CI "
      f"[{R['ci_usfr_bil_lo']:+.3f}, {R['ci_usfr_bil_hi']:+.3f}] "
      f"({R['ci_usfr_bil_neg']:.1f}% of draws < 0)")
print(f"                        USFR-SHY 95% CI "
      f"[{R['ci_usfr_shy_lo']:+.3f}, {R['ci_usfr_shy_hi']:+.3f}] "
      f"({R['ci_usfr_shy_neg']:.1f}% < 0)")

USFR-BIL   +0.160%/yr   HAC t = +0.68
TFLO-BIL   +0.198%/yr   HAC t = +0.96
USFR-SHY   +0.435%/yr   HAC t = +0.96
BIL-SHY    +0.275%/yr   HAC t = +0.72
USFR-TFLO  -0.038%/yr   HAC t = -0.13

bootstrap (21d blocks): USFR-BIL 95% CI [-0.244, +0.548] (21.4% of draws < 0)
                        USFR-SHY 95% CI [-0.432, +1.321] (15.0% < 0)


## 3. Launch liquidity — a data-quality cut, not a regime cut

USFR and TFLO listed in Feb-2014 and barely traded for years: USFR's quoted vol was 2.7% annualised in 2014 against 0.28% in 2025, and TFLO printed a +466 bp day next to a −446 bp day in Dec-2014. The *point estimate* of the pickup is stable across the split; only the noise shrinks. One cell (TFLO−BIL post-2018) touches *t* = 2 — one cell among many is not a rejection, and we do not treat it as one.

> 💡 **In plain words:** the early prices are stale quotes on a fund nobody was trading, not losses anyone suffered.

In [4]:
print(f"2014-2017  USFR-BIL {R['early_usfr_bil']:+.3f}%/yr (t={R['early_t']:+.2f})")
print(f"2018-2026  USFR-BIL {R['late_usfr_bil']:+.3f}%/yr (t={R['late_t']:+.2f})   "
      f"TFLO-BIL {R['late_tflo_bil']:+.3f}%/yr (t={R['late_tflo_t']:+.2f})   "
      f"USFR-SHY {R['late_usfr_shy']:+.3f}%/yr (t={R['late_usfr_shy_t']:+.2f})")

2014-2017  USFR-BIL +0.169%/yr (t=+0.23)
2018-2026  USFR-BIL +0.155%/yr (t=+1.40)   TFLO-BIL +0.143%/yr (t=+2.08)   USFR-SHY +0.693%/yr (t=+1.24)


## 4. Regime cut on the direction of ^IRX

Label = sign of the 63-day change in the 13-week bill rate through *t*, dead band ±0.25 pp, applied at *t*+1.

In [5]:
rows = [('rising',  R['reg_rise_n'], R['rise_usfr'], R['rise_bil'], R['rise_shy'], R['rise_gap'], R['rise_gap_t']),
        ('flat',    R['reg_flat_n'], R['flat_usfr'], R['flat_bil'], R['flat_shy'], R['flat_gap'], R['flat_gap_t']),
        ('falling', R['reg_fall_n'], R['fall_usfr'], R['fall_bil'], R['fall_shy'], R['fall_gap'], R['fall_gap_t'])]
print(f"{'regime':8s} {'n':>5s} {'USFR':>7s} {'BIL':>7s} {'SHY':>7s} {'USFR-SHY':>10s} {'t':>7s}")
for r_, n, u, b, s, g, t in rows:
    print(f'{r_:8s} {n:5d} {u:+6.2f}% {b:+6.2f}% {s:+6.2f}% {g:+9.2f}% {t:+7.2f}')

regime       n    USFR     BIL     SHY   USFR-SHY       t
rising     626  +2.65%  +2.12%  -0.04%     +2.69%   +1.94
flat      2038  +1.55%  +1.49%  +1.73%     -0.18%   -0.35
falling    390  +3.14%  +2.84%  +2.87%     +0.26%   +0.23


## 5. The headline test — one HAC-OLS regression, not a walk through sub-samples

Regress the daily difference (annualised pp) on rising- and falling-rate dummies over the **whole** sample. The **contrast** = advantage when rates rise minus advantage when they fall. Right sign, large magnitude, *t* short of 2.

> 💡 **In plain words:** the floater wins about 3 percentage points a year more when rates are climbing than when they are falling — but the sample cannot rule out that this is luck.

In [6]:
print(f"USFR-SHY: flat {R['flat_gap']:+.2f}%   rising extra {R['rising_extra']:+.2f} "
      f"(t={R['rising_extra_t']:+.2f})   falling extra {R['falling_extra']:+.2f} "
      f"(t={R['falling_extra_t']:+.2f})")
print(f"  -> contrast {R['contrast']:+.2f} pp/yr (HAC t = {R['contrast_t']:+.2f})")
print(f"BIL-SHY : contrast {R['contrast_bil']:+.2f} pp/yr (t = {R['contrast_bil_t']:+.2f})")
print(f"\nclassifier sweep ({R['sweep_total']} window/dead-band settings): "
      f"positive in {R['sweep_positive']}/{R['sweep_total']}, range "
      f"{R['sweep_lo']:+.2f} to {R['sweep_hi']:+.2f} pp/yr, |t|>=2 in "
      f"{R['sweep_fire']}/{R['sweep_total']} (closest {R['sweep_best']:+.2f}, t={R['sweep_best_t']:+.2f})")

USFR-SHY: flat -0.18%   rising extra +2.87 (t=+1.92)   falling extra +0.45 (t=+0.37)
  -> contrast +2.43 pp/yr (HAC t = +1.37)
BIL-SHY : contrast +2.19 pp/yr (t = +1.33)

classifier sweep (12 window/dead-band settings): positive in 12/12, range +0.03 to +6.41 pp/yr, |t|>=2 in 0/12 (closest +2.69, t=+1.94)


## 6. Cycle windows and the duration attribution

The windows are a declared **ASSUMPTION** (a hardcoded Fed calendar); the ^IRX cut above is the mechanical alternative and tells the same story. The attribution compares the realised cumulative USFR−SHY gap with the textbook *D × Δy*, where *D* ≈ 1.85 is SHY's published effective duration (an assumption, swept 1.60-2.10) and Δy is proxied by the **13-week** rate — deliberately crude, and the residual is exactly the curve.

In [7]:
print(f"{'window':16s} {'USFR':>7s} {'BIL':>7s} {'SHY':>7s} {'USFR-SHY':>10s} {'t':>7s}")
for w, u, b, s, g, t in [('ZIRP 2014-21', R['zirp_usfr'], R['zirp_bil'], R['zirp_shy'], R['zirp_gap'], R['zirp_gap_t']),
                         ('hiking 22-23', R['hike_usfr'], R['hike_bil'], R['hike_shy'], R['hike_gap'], R['hike_gap_t']),
                         ('plateau 23-24', R['plat_usfr'], R['plat_bil'], R['plat_shy'], R['plat_gap'], R['plat_gap_t']),
                         ('cutting 24-26', R['cut_usfr'], R['cut_bil'], R['cut_shy'], R['cut_gap'], R['cut_gap_t'])]:
    print(f'{w:16s} {u:+6.2f}% {b:+6.2f}% {s:+6.2f}% {g:+9.2f}% {t:+7.2f}')
print(f"\nhiking : d(13w) {R['hike_dirx']:+.2f} pp -> predicted gap "
      f"{R['hike_pred']:+.2f}%, realised {R['hike_gap_cum']:+.2f}% (curve inverted)")
print(f"cutting: d(13w) {R['cut_dirx']:+.2f} pp -> predicted gap "
      f"{R['cut_pred']:+.2f}%, realised {R['cut_gap_cum']:+.2f}%  <- duration never paid back")
print(f"also: USFR-BIL in the hiking window {R['hike_usfr_bil']:+.2f}%/yr "
      f"(t={R['hike_usfr_bil_t']:+.2f}) on {R['hike_n']} days - the reset-speed effect")

window              USFR     BIL     SHY   USFR-SHY       t
ZIRP 2014-21      +0.73%  +0.58%  +1.11%     -0.38%   -0.80
hiking 22-23      +3.39%  +2.85%  -1.07%     +4.46%   +1.99
plateau 23-24     +4.83%  +5.24%  +5.50%     -0.66%   -0.40
cutting 24-26     +4.24%  +4.07%  +3.39%     +0.86%   +0.87

hiking : d(13w) +4.94 pp -> predicted gap +9.13%, realised +6.45% (curve inverted)
cutting: d(13w) -1.24 pp -> predicted gap -2.29%, realised +1.68%  <- duration never paid back
also: USFR-BIL in the hiking window +0.54%/yr (t=+2.05) on 356 days - the reset-speed effect


## 7. Drawdowns, costs and borrow — what survives contact with reality

Held sleeves pay a round trip once, so friction is amortised: even a punitive 5 bp spread leaves the USFR−SHY gap positive. Read instead as a dollar-neutral pair, the same difference is **dead by 50 bps of borrow**. The durable difference is the drawdown, and it costs nothing.

In [8]:
print(f"drawdowns 2018+: USFR {R['usfr_dd_liq']:+.2f}%  TFLO {R['tflo_dd_liq']:+.2f}%  "
      f"BIL {R['bil_dd_liq']:+.2f}%  SHY {R['shy_dd_liq']:+.2f}%  "
      f"(vol {R['usfr_vol_liq']:.2f}% vs {R['shy_vol_liq']:.2f}%)")
print(f"held sleeve, USFR-SHY {R['usfr_shy']:+.3f}%/yr gross -> "
      f"{R['cost1_1y']:+.3f}% at 1bp/1yr, {R['cost5_1y']:+.3f}% at 5bp/1yr, "
      f"{R['cost5_3y']:+.3f}% at 5bp/3yr")
print(f"long/short pair       : {R['borrow25']:+.3f}% at 25bp borrow, "
      f"{R['borrow50']:+.3f}% at 50bp, {R['borrow100']:+.3f}% at 100bp")

drawdowns 2018+: USFR -0.40%  TFLO -0.16%  BIL -0.21%  SHY -5.71%  (vol 0.60% vs 1.67%)
held sleeve, USFR-SHY +0.435%/yr gross -> +0.395% at 1bp/1yr, +0.235% at 5bp/1yr, +0.368% at 5bp/3yr
long/short pair       : +0.185% at 25bp borrow, -0.065% at 50bp, -0.565% at 100bp


## 8. Live synthetic control — the machinery is unbiased

Same rate cycle in both worlds; only the fixed leg's duration changes. Planted (D = 1.85): the contrast must be large and positive, with the floater ahead when rates rise and behind when they fall. Null (D = 0): the classifier is just as busy, but there is nothing to find. This proves the real-tape *t* = 1.37 is a sample-size fact, not a broken harness.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from frn_front import data, strategy as st
pl = st.synthetic_detect(data.synthetic_panel(signal_strength=1.0, seed=922)[0])
print(f"planted (D=1.85): contrast {pl['contrast']:+.2f} pp/yr (t={pl['contrast_t']:+.1f}); "
      f"rising {pl['rising_extra']:+.2f}, falling {pl['falling_extra']:+.2f}")
nl = np.array([st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=922+s)[0])['contrast'] for s in range(8)])
tn = np.array([st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=922+s)[0])['contrast_t'] for s in range(8)])
print(f"null x8 (D=0)   : contrast mean {nl.mean():+.2f} (sd {nl.std(ddof=1):.2f}), "
      f"|t|>=2 in {(abs(tn)>=2).sum()}/8")
half = st.synthetic_detect(data.synthetic_panel(signal_strength=0.5, seed=922)[0])
print(f"half duration   : contrast {half['contrast']:+.2f} pp/yr -> the estimator scales with the planted effect")

planted (D=1.85): contrast +8.35 pp/yr (t=+14.2); rising +5.43, falling -2.92


null x8 (D=0)   : contrast mean +0.14 (sd 0.40), |t|>=2 in 0/8
half duration   : contrast +4.52 pp/yr -> the estimator scales with the planted effect


## Verdict

- **Signal — Weak.** Correct sign everywhere and economically large — floaters beat 1-3y fixed by +4.46 pp/yr through the hikes, lose by 0.38-0.66 pp/yr when rates sit still, and the rising-minus-falling contrast is positive in 12/12 classifier settings. But nothing is robust at |*t*| = 2: headline contrast +2.43 pp/yr (*t* = +1.37), unconditional pairs *t* = +0.68 to +0.96, all bootstrap CIs straddling zero, and the contrast clears 2 in 0/12 classifier settings. The ranking flips with the regime and that flip *is* the finding — but no regime earns a stamp of its own, so this is Weak (a mechanism the tape cannot certify), not Mixed (a verdict that splits into stamps).
- **Tradability — Fragile.** The floater-over-bills pickup (+0.160%/yr) is stable and cost-proof but fee-sized; the floater-over-SHY choice is a rate call; the pair dies at -0.065%/yr on 50 bps of borrow. The bankable part is risk, not return: -0.4% worst drawdown against -5.7% for the same total return.
- **Non-tape inputs**, all declared and swept: the ^IRX cash-accrual convention (252 vs 360), the hardcoded Fed-cycle calendar, the regime window and dead band, and SHY's assumed 1.85-year effective duration.